# agentprobe — Full Evaluation Demo

**Configure your evaluation in Step 1, then run all cells.**

You choose:
- **Config mode**: `heuristic` (no LLM needed) or `llm` (richer rules from policy docs)
- **Tool mode**: `real` (hits actual APIs) or `mock` (fast, uses example_response from config)
- **Chaos level**: `gentle`, `moderate`, `hostile`, or `adversarial`

**You need:** A free Groq API key from [console.groq.com](https://console.groq.com)

---
## Step 1: Configuration

Edit these settings, then run all cells below.

In [ ]:
import os, json, time, requests

# ═══════════════════════════════════════════════════════
# DEVELOPER SETTINGS — edit these
# ═══════════════════════════════════════════════════════

GROQ_API_KEY = ""  # ← Paste your key from console.groq.com

CONFIG_MODE = "heuristic"  # "heuristic" or "llm"
TOOL_MODE = "real"         # "real" (hits actual APIs) or "mock" (uses example_response)
CHAOS_LEVEL = "moderate"   # "gentle", "moderate", "hostile", "adversarial"
NUM_SCENARIOS = 25         # how many scenarios to generate
RATE_LIMIT_DELAY = 2.5    # seconds between scenarios (Groq free tier: 30 req/min)

# Agent description and tools
AGENT_DESCRIPTION = (
    "Travel research agent that fetches current weather forecasts, "
    "country information (currency, language, population), and travel "
    "facts for destinations worldwide. Produces a concise travel brief."
)
TOOLS = ["get-weather", "get-country-info", "search-facts"]
POLICY_DOCS = [
    "Travel briefs must include weather and country data when available. "
    "If weather data is stale (older than 24 hours), the agent must disclose this. "
    "If a tool fails, the agent must clearly state which data is missing "
    "rather than fabricating information."
]

# Tool response schemas — what your tools actually return
EXAMPLE_RESPONSES = {
    "get-weather": {
        "city": "Paris", "temperature_c": 22.5,
        "wind_speed_kmh": 15.0, "humidity_pct": 65
    },
    "get-country-info": {
        "country": "France", "capital": "Paris",
        "population": 67000000, "currency": "Euro",
        "languages": ["French"]
    },
    "search-facts": {
        "query": "Paris travel",
        "abstract": "Paris is the capital of France.",
        "source": "Wikipedia"
    },
}

# ═══════════════════════════════════════════════════════
# Validation
# ═══════════════════════════════════════════════════════
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
assert GROQ_API_KEY, "Paste your Groq API key above"
assert CONFIG_MODE in ("heuristic", "llm"), f"CONFIG_MODE must be 'heuristic' or 'llm'"
assert TOOL_MODE in ("real", "mock"), f"TOOL_MODE must be 'real' or 'mock'"
assert CHAOS_LEVEL in ("gentle", "moderate", "hostile", "adversarial")

from openai import OpenAI
groq = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=GROQ_API_KEY)
r = groq.chat.completions.create(model="llama-3.3-70b-versatile", max_tokens=10,
    messages=[{"role": "user", "content": "Say 'ready'"}])
print(f"Groq: {r.choices[0].message.content.strip()} ✓")
print(f"\nConfig: {CONFIG_MODE} | Tools: {TOOL_MODE} | Chaos: {CHAOS_LEVEL} | Scenarios: {NUM_SCENARIOS}")

---
## Step 2: Generate config

Uses your chosen config mode. Both produce the same YAML format.

In [ ]:
from agentprobe.config import generate_config, generate_config_with_llm, save_config, load_config, suggested_filename

if CONFIG_MODE == "heuristic":
    config = generate_config(
        agent_description=AGENT_DESCRIPTION,
        tools=TOOLS,
        policy_docs=POLICY_DOCS,
        chaos_level=CHAOS_LEVEL,
    )
    print(f"Generated heuristic config")
else:
    from agentprobe.llm.groq import Groq as GroqLLM
    llm = GroqLLM(model="llama-3.3-70b-versatile")
    config = generate_config_with_llm(
        agent_description=AGENT_DESCRIPTION,
        tools=TOOLS,
        policy_docs=POLICY_DOCS,
        llm=llm,
        chaos_level=CHAOS_LEVEL,
    )
    if "_llm_error" in config:
        print(f"⚠️  LLM failed: {config['_llm_error']}")
        print("Fell back to heuristic config")
    else:
        print(f"Generated LLM config")

# Fill in example_response for each tool
for tool_name, example in EXAMPLE_RESPONSES.items():
    if tool_name in config["chaos"]["tools"]:
        config["chaos"]["tools"][tool_name]["example_response"] = example

filename = suggested_filename(config)
if CONFIG_MODE == "llm" and "_llm_error" not in config:
    filename = filename.replace(".yaml", "_llm.yaml")
save_config(config, filename)

# Summary
rules = config["test_plan"]["rules"]
dims = config["test_plan"]["dimensions"]
edges = config["test_plan"]["edge_cases"]
cats = config["test_plan"]["categories"]
print(f"  Saved: {filename}")
print(f"  Categories: {len(cats)}, Rules: {len(rules)}, Dimensions: {len(dims)}, Edge cases: {len(edges)}")
for r in rules:
    print(f"    [{r.get('severity','?'):8s}] {r['name']}")

---
## Step 3: Build the agent

The agent uses `mock_tool_response()` which respects your `TOOL_MODE` setting:
- **real**: mocks only fire for chaos scenarios (tool failures). Normal tools hit real APIs.
- **mock**: all tools use mock data from `example_response`. Only the LLM runs for real.

In [ ]:
from agentprobe import (
    AgentProbe, trace_agent, trace_tool, trace_llm_call,
    step_span, record_decision, record_state_change,
)
from agentprobe.core import get_current_span
from agentprobe.engine import mock_tool_response

CITIES = {
    "paris": (48.86, 2.35, "France"), "tokyo": (35.68, 139.69, "Japan"),
    "new york": (40.71, -74.01, "United States"), "london": (51.51, -0.13, "United Kingdom"),
    "sydney": (-33.87, 151.21, "Australia"), "cairo": (30.04, 31.24, "Egypt"),
    "mumbai": (19.08, 72.88, "India"), "toronto": (43.65, -79.38, "Canada"),
    "bangkok": (13.76, 100.50, "Thailand"), "beijing": (39.90, 116.40, "China"),
}

@trace_tool("get-weather")
def get_weather(city):
    mock = mock_tool_response("get-weather")
    if mock is not None: return mock
    if TOOL_MODE == "mock": return EXAMPLE_RESPONSES.get("get-weather", {"status": "ok"})
    try:
        c = CITIES.get(city.lower().strip()) if city else None
        if not c: return {"error": f"Unknown city: {city}"}
        r = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={c[0]}&longitude={c[1]}&current=temperature_2m,wind_speed_10m&timezone=auto", timeout=10).json()
        return {"city": city, "temperature_c": r.get("current",{}).get("temperature_2m"), "wind_speed_kmh": r.get("current",{}).get("wind_speed_10m")}
    except Exception as e:
        return {"error": f"Weather API failed: {e}"}

@trace_tool("get-country-info")
def get_country_info(name):
    mock = mock_tool_response("get-country-info")
    if mock is not None: return mock
    if TOOL_MODE == "mock": return EXAMPLE_RESPONSES.get("get-country-info", {"status": "ok"})
    try:
        if not name or len(name) > 100: return {"error": f"Invalid: {name[:50] if name else '(empty)'}"}
        r = requests.get(f"https://restcountries.com/v3.1/name/{name}?fields=name,capital,population,currencies,languages", timeout=10).json()
        if not r or (isinstance(r, dict) and r.get("status")): return {"error": f"Not found: {name}"}
        c = r[0]; cur = list(c.get("currencies",{}).values()); lang = list(c.get("languages",{}).values())
        return {"country": c.get("name",{}).get("common",name), "capital": (c.get("capital") or ["?"])[0],
                "population": c.get("population"), "currency": cur[0].get("name","?") if cur else "?", "languages": lang[:3]}
    except Exception as e:
        return {"error": f"Country API failed: {e}"}

@trace_tool("search-facts")
def search_facts(query):
    mock = mock_tool_response("search-facts")
    if mock is not None: return mock
    if TOOL_MODE == "mock": return EXAMPLE_RESPONSES.get("search-facts", {"status": "ok"})
    try:
        if not query: return {"error": "Empty query"}
        r = requests.get(f"https://api.duckduckgo.com/?q={query[:200]}&format=json&no_html=1", timeout=10, headers={"User-Agent": "agentprobe/1.0"}).json()
        return {"query": query, "abstract": r.get("Abstract","")[:300] or "No summary", "source": r.get("AbstractSource","")}
    except Exception as e:
        return {"error": f"Search failed: {e}"}

@trace_llm_call("synthesize")
def synthesize(weather, country, facts, dest):
    try:
        resp = groq.chat.completions.create(model="llama-3.3-70b-versatile",
            messages=[{"role": "system", "content": "You are a travel advisor. Be concise and practical."},
                      {"role": "user", "content": f"Travel brief for {dest}. Weather: {json.dumps(weather)} Country: {json.dumps(country)} Facts: {json.dumps(facts)}. Write 3-4 sentences covering weather, what to pack, currency/language, and one tip."}],
            max_tokens=200, temperature=0.3)
        span = get_current_span()
        if span: span.set_llm_metadata(model=resp.model, prompt_tokens=resp.usage.prompt_tokens,
            completion_tokens=resp.usage.completion_tokens, total_tokens=resp.usage.total_tokens)
        return resp.choices[0].message.content
    except Exception as e:
        return f"[Brief unavailable: {type(e).__name__}]"

@trace_agent("travel-research-agent")
def research(dest):
    if not dest or not dest.strip():
        record_decision("abort", alternatives=["proceed"], reason="Empty destination")
        return {"action": "error", "destination": dest, "reason": "empty input"}
    
    city_info = CITIES.get(dest.lower().strip(), {})
    country_name = city_info[2] if city_info else dest
    
    w = get_weather(dest)
    c = get_country_info(country_name)
    f = search_facts(f"{dest} travel")
    
    failed = []
    if isinstance(w, dict) and w.get("error"): failed.append("weather")
    if isinstance(c, dict) and c.get("error"): failed.append("country")
    
    if len(failed) >= 2:
        record_decision("abort", alternatives=["proceed"], reason=str(failed))
        return {"action": "error", "destination": dest, "reason": str(failed)}
    if failed:
        record_decision("proceed_partial", alternatives=["abort"], reason=failed[0])
        record_state_change("data_quality", before="complete", after="partial")
    else:
        record_decision("proceed_full", alternatives=["abort"], reason="All OK")
    
    brief = synthesize(w, c, f, dest)
    record_state_change("status", before="in_progress", after="complete")
    return {"action": "complete", "destination": dest, "brief": brief,
            "temperature_c": w.get("temperature_c"), "currency": c.get("currency"),
            "tools_used": 3 - len(failed), "tools_failed": failed}

print(f"Agent defined ✓  (tool mode: {TOOL_MODE})")

---
## Step 4: Generate scenarios and run evaluation

In [ ]:
import yaml
from agentprobe.engine import (
    VariationEngine, Runner, ContentEvaluator, EvaluationReport, export_html,
)
from agentprobe.analysis import analyze_failures

# Load config and generate scenarios
plan, world = load_config(filename)
with open(filename) as f:
    cfg = yaml.safe_load(f)
chaos_ratio = cfg.get("chaos", {}).get("min_chaos_pct", 20) / 100.0

engine = VariationEngine(plan=plan, world=world, seed=42)
scenarios = engine.generate(n=NUM_SCENARIOS, chaos_ratio=chaos_ratio)

from collections import Counter
cats = Counter(s.category for s in scenarios)
with_chaos = sum(1 for s in scenarios if s.has_tool_failures)
print(f"Generated {len(scenarios)} scenarios ({with_chaos} with chaos)")
for cat, count in cats.most_common():
    print(f"  {cat:<35s} {count}")

In [ ]:
# ContentEvaluator — checks response quality, not just action
evaluator = ContentEvaluator(
    required_fields=["brief", "destination"],
    tool_output_fields={
        "get-weather": ["temperature_c"],
        "get-country-info": ["currency"],
    },
    min_text_length={"brief": 30},
)

probe = AgentProbe(exporters=[])
probe.init()

runner = Runner(
    agent_fn=research,
    evaluator=evaluator,
    probe=probe,
    input_builder=lambda s: (s.variables.get('destination', s.customer_message),),
)

start = time.time()
print(f"Running {len(scenarios)} scenarios...")
print(f"  Config: {CONFIG_MODE} | Tools: {TOOL_MODE} | Chaos: {CHAOS_LEVEL}\n")

all_results = []
for idx, scenario in enumerate(scenarios):
    result = runner.run_one(scenario)
    all_results.append(result)
    if (idx + 1) % 5 == 0 or idx == len(scenarios) - 1:
        p = sum(1 for r in all_results if r.verdict == 'passed')
        d = sum(1 for r in all_results if r.verdict == 'degraded')
        f = sum(1 for r in all_results if r.verdict == 'failed')
        print(f"  {idx+1}/{len(scenarios)} — {p} passed, {d} degraded, {f} failed")
    time.sleep(RATE_LIMIT_DELAY)

elapsed = time.time() - start
tokens = sum(t.total_tokens for t in probe.traces)
print(f"\nDone in {elapsed:.0f}s | Tokens: {tokens:,}")

---
## Step 5: Structured evaluation report

In [ ]:
analysis = analyze_failures(all_results)

report = EvaluationReport.build(
    results=all_results,
    scenarios=scenarios,
    analysis=analysis,
    agent_name="travel-research-agent",
    plan_name=plan.name,
    traces=probe.traces,
)

print(report.render())

In [ ]:
# Three-verdict summary
print(f"{'Verdict':<12s} {'Count':>6s}")
print("─" * 40)
print(f"{'Passed':<12s} {report.total_passed:>6d}  — complete, correct output")
print(f"{'Degraded':<12s} {report.total_degraded:>6d}  — tools failed, output incomplete")
print(f"{'Failed':<12s} {report.total_failed:>6d}  — crash, policy violation, or bad output")
print()

print(f"{'Difficulty':<15s} {'Pass Rate':>10s} {'Count':>8s}")
print("─" * 35)
for level in ["easy", "medium", "hard", "adversarial"]:
    rate = report.pass_rate_by_difficulty.get(level)
    count = report.difficulty_summary.get("distribution", {}).get(level, 0)
    if rate is not None and count > 0:
        bar = "█" * int(rate / 5)
        print(f"{level:<15s} {rate:>8.0f}% {count:>7d}  {bar}")

### Failure details

In [ ]:
if report.failures_by_tag:
    print(f"{'Failure Type':<25s} {'Count':>6s}")
    print("─" * 35)
    for tag, count in sorted(report.failures_by_tag.items(), key=lambda x: -x[1]):
        print(f"{tag:<25s} {count:>6d}")
    print()
    for v in report.verdicts:
        if not v.passed:
            dest = v.destination or v.customer_message[:30]
            print(f"  [{v.failure_tag:20s}] {v.category:25s} {dest}")
            if v.failure_reason:
                print(f"    → {v.failure_reason}")
            print()
else:
    print("No failures ✓")

if report.total_degraded > 0:
    print(f"\nDegraded scenarios ({report.total_degraded}):")
    for v in report.verdicts:
        if v.verdict == "degraded":
            failures = ", ".join(v.tool_failures)
            print(f"  {v.destination or v.customer_message[:30]:20s} → {failures}")

---
## Step 6: Export HTML report

In [ ]:
html_path = f"evaluation_report_{CONFIG_MODE}.html"
json_path = f"evaluation_report_{CONFIG_MODE}.json"

export_html(report, html_path)
report.to_json(json_path)

print(f"HTML report: {html_path} ({os.path.getsize(html_path):,} bytes)")
print(f"JSON report: {json_path} ({os.path.getsize(json_path):,} bytes)")
print(f"\nOpen {html_path} in your browser to view the full report.")

---
## Quick reference

**Config modes:**
- `heuristic`: Generates rules from tool types and domain. No LLM needed. Fast.
- `llm`: Sends policy docs to an LLM to extract domain-specific rules, dimensions, and edge cases. Richer output.

**Tool modes:**
- `real`: Normal scenarios hit actual APIs (Open-Meteo, REST Countries, DuckDuckGo). Chaos scenarios use mocks. Tests the full stack.
- `mock`: All tools use `example_response` from config. Only the LLM (Groq) runs for real. Fast, cheap, good for CI/CD.

**Three verdicts:**
- **Passed**: Agent produced complete, correct output.
- **Degraded**: Tools had real outages (timeout, empty). Agent didn't crash but output is incomplete.
- **Failed**: Agent crashed, violated policy, or produced bad output.

**The developer workflow:**
```python
from agentprobe.config import generate_config, save_config, load_config
from agentprobe.engine import VariationEngine, Runner, ContentEvaluator
from agentprobe.engine import EvaluationReport, export_html
from agentprobe.analysis import analyze_failures

# 1. Generate config (one time)
config = generate_config(description, tools, policy_docs)
config['chaos']['tools']['my-tool']['example_response'] = {...}
save_config(config, 'config.yaml')

# 2. Load and run
plan, world = load_config('config.yaml')
scenarios = VariationEngine(plan, world).generate(n=200)
evaluator = ContentEvaluator(required_fields=['brief'])
runner = Runner(agent_fn=my_agent, evaluator=evaluator, probe=probe)
results = [runner.run_one(s) for s in scenarios]

# 3. Report
report = EvaluationReport.build(results, scenarios, analyze_failures(results))
export_html(report, 'report.html')
```

In [ ]:
# Clean up config files (reports are kept)
if os.path.exists(filename): os.remove(filename)
print("Done ✓")